# Qwen2.5 Question Generation Smoke Test

Validates `Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8` as a replacement q_generator for LLaMA 70B on the PreciseWikiQA task.

**Requires:** REMOTE_GPU context (H100). The server will auto-detect the GPTQ model and load it via vLLM.

**What this tests:**
1. `model_map` lookup succeeds (no `KeyError`)
2. vLLM server starts and loads the GPTQ model
3. Questions are generated and pass the answerability filter
4. Output quality looks reasonable

In [1]:
import os
repo_root = "/mnt/home/hyang1/LLM_research/HalluLens"
os.chdir(repo_root)
print(f"cwd: {os.getcwd()}")


cwd: /mnt/home/hyang1/LLM_research/HalluLens


In [2]:
from scripts.run_with_server import run_experiment

## Configuration

Adjust `Q_GENERATOR` to test different Qwen2.5 variants:
- `Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8` — primary target (GPTQ, ~72GB)
- `Qwen/Qwen2.5-14B-Instruct` — faster/cheaper option
- `Qwen/Qwen2.5-7B-Instruct` — fastest option

In [3]:
# Primary target
Q_GENERATOR = "Qwen/Qwen2.5-72B-Instruct-GPTQ-Int8"

# Inference model (only needed for inference/all steps; not used during generate)
INFERENCE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# N=5 with quick_debug_mode=True uses the first 5 wiki docs — fast smoke test
N = 5

## Run: Generate QA pairs

This starts the vLLM server with the Qwen2.5 model, generates `N` question-answer pairs, and writes them to:
```
data/precise_qa/save/qa_goodwiki_Llama-3.1-8B-Instruct_dynamic.jsonl
```

In [ ]:
run_experiment(
    step="generate",
    task="precisewikiqa",
    model=INFERENCE_MODEL,
    q_generator=Q_GENERATOR,
    N=N,
    quick_debug_mode=True,
    server_startup_timeout=1800,
    max_model_len=2048,
)

2026-03-12 18:05:10 | ERROR    | utils.lm:_wait_for_server:277 - Server process died. STDOUT: None, STDERR: None


RuntimeError: Server process terminated unexpectedly

In [5]:
!nvidia-smi

Thu Mar 12 18:49:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:18:00.0 Off |                    0 |
| N/A   32C    P0             69W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Inspect output

In [ ]:
import jsonlines
import pandas as pd

# Default output path (matches precise_wikiqa.py logic)
model_name = INFERENCE_MODEL.split("/")[-1]
qa_path = project_root / f"data/precise_qa/save/qa_goodwiki_{model_name}_dynamic.jsonl"

print(f"QA output path: {qa_path}")
print(f"Exists: {qa_path.exists()}")

In [ ]:
if qa_path.exists():
    qa_pairs = list(jsonlines.open(qa_path))
    df = pd.DataFrame(qa_pairs)
    print(f"Total QA pairs: {len(df)}")
    print(f"Columns: {list(df.columns)}\n")
    
    # Show questions and answers
    for i, row in df.head(N).iterrows():
        print(f"[{i}] Q: {row.get('question', row.get('prompt', '?'))}")
        print(f"     A: {row.get('answer', '?')}")
        print(f"     Title: {row.get('title', '?')}")
        print()
else:
    print("QA file not found — check if generation completed successfully above.")

## Quality checks

Quick sanity checks on the generated questions.

In [ ]:
if qa_path.exists() and len(df) > 0:
    q_col = 'question' if 'question' in df.columns else 'prompt'
    a_col = 'answer'

    end_with_q     = df[q_col].str.strip().str.endswith('?').sum()
    non_empty_ans  = df[a_col].notna().sum() if a_col in df.columns else 0
    short_ans      = (df[a_col].str.split().str.len() <= 10).sum() if a_col in df.columns else 0

    print(f"Questions ending in '?':   {end_with_q}/{len(df)}")
    print(f"Non-empty answers:         {non_empty_ans}/{len(df)}")
    print(f"Answers ≤10 words (precise): {short_ans}/{len(df)}")

## (Optional) Test smaller Qwen2.5 variants

Uncomment to compare generation quality/speed against smaller variants.

In [ ]:
# for variant in ["Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-7B-Instruct"]:
#     print(f"\n{'='*60}")
#     print(f"Testing: {variant}")
#     print('='*60)
#     run_experiment(
#         step="generate",
#         task="precisewikiqa",
#         model=INFERENCE_MODEL,
#         q_generator=variant,
#         N=N,
#         quick_debug_mode=True,
#     )